# 02 Cleaning and Feature Engineering

This notebook explains and runs the first cleaning stage for the Kaggle Home Credit application data. The goal is to create a cleaner applicant-level table that can be used for SQL analysis, Tableau dashboarding, and machine learning.

## Cleaning Goals

1. Keep one row per applicant.
2. Rename raw Kaggle columns into business-friendly names.
3. Convert negative day fields into readable year fields.
4. Handle known special placeholder values.
5. Create missing-value flags where missingness may be informative.
6. Create early affordability and risk features.
7. Export a clean processed table for SQL, Tableau, and modelling.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from create_application_base import build_application_base, OUTPUT_FILE, CLEANING_REPORT

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)

## 1. Build Clean Applicant Table

The function below loads `application_train.csv`, applies the cleaning decisions, and returns a clean applicant-level dataset.

In [ ]:
clean_df = build_application_base()
clean_df.shape

In [ ]:
clean_df.head()

## 2. Validate Applicant Grain

For modelling, SQL, and Tableau, this table should have exactly one row per applicant.

In [ ]:
grain_check = {
    'rows': len(clean_df),
    'unique_applicants': clean_df['applicant_id'].nunique(),
    'duplicate_applicant_rows': len(clean_df) - clean_df['applicant_id'].nunique(),
}
grain_check

## 3. Validate Target Balance

The default class is expected to be much smaller than the non-default class.

In [ ]:
target_summary = (
    clean_df['target_default']
    .value_counts(dropna=False)
    .rename_axis('target_default')
    .reset_index(name='applicant_count')
)
target_summary['applicant_percent'] = (
    target_summary['applicant_count'] / target_summary['applicant_count'].sum() * 100
).round(2)
target_summary

## 4. Check Engineered Features

The features below translate raw loan fields into business concepts: age, employment stability, repayment burden, loan-to-income ratio, income band, and risk band.

In [ ]:
feature_columns = [
    'age_years',
    'employment_years',
    'repayment_burden_ratio',
    'loan_to_income_ratio',
    'income_band',
    'external_score_mean',
    'risk_proxy_score',
    'risk_band',
]
clean_df[feature_columns].describe(include='all')

## 5. Missing Value Review

After the first cleaning pass, remaining missing values are expected mainly in fields where absence can be meaningful, such as external scores or employment tenure.

In [ ]:
missing_summary = (
    clean_df.isna().sum().rename('missing_count').reset_index().rename(columns={'index': 'column'})
)
missing_summary['missing_percent'] = (
    missing_summary['missing_count'] / len(clean_df) * 100
).round(2)
missing_summary.sort_values('missing_percent', ascending=False).head(15)

## 6. Export Clean Data

The processed CSV is created locally for analysis. It is intentionally ignored by Git because it is a large derived data file.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(OUTPUT_FILE, index=False)
OUTPUT_FILE

## 7. Cleaning Notes For Portfolio

The cleaning strategy is documented in `docs/cleaning_strategy.md`, and the generated cleaning summary is saved in `reports/cleaning_summary.md`.